# Flujo compresible en tuberías con fricción (gas ideal)

En este notebook estudiamos el flujo unidimensional de un gas ideal en tuberías horizontales con pérdida de carga por fricción, usando como referencia el formulario:

> **“Fórmulas de flujo compresible y Lapple”**

Nos centraremos en dos modelos ideales:

- **Modelo isotérmico**: el gas se mantiene a temperatura constante a lo largo de la tubería.
- **Modelo adiabático**: el flujo es adiabático y no hay intercambio de calor neto con el entorno.

La idea es:

1. Formular los balances y las ecuaciones adimensionales según el formulario del curso.
2. Implementar funciones en Julia que **resuelvan numéricamente** los sistemas de ecuaciones no lineales (como hace la planilla Calc).
3. Aplicar esas funciones a ejercicios concretos (problemas de parcial, guía, etc.).

In [2]:
## Paquetes a utilizar

using NLsolve        # Para resolver sistemas no lineales
using Printf         # Para formatear salidas numéricas

# Si más adelante queremos usar Unitful:
# using Unitful

In [3]:
# Área de la sección de una tubería circular
area_tubo(D) = π * D^2 / 4

# Densidad de gas ideal
ρ_ideal(P, T, R) = P / (R * T)

# Número de Reynolds
reynolds_number(ρ, v, D, μ) = ρ * v * D / μ

"""
    friction_factor_churchill(Re, ε, D)

Devuelve el factor de fricción Darcy-Weisbach f usando
la correlación de Churchill (válida en la práctica para
toda la gama de Re y rugosidades).
"""
function friction_factor_churchill(Re, ε, D)
    # Churchill 1977, correlación continua
    A = (2.457 * log( (7 / Re)^0.9 + 0.27 * (ε / D) ))^16
    B = (37530 / Re)^16
    f = 8 * ((8 / Re)^12 + 1 / (A + B)^(3/2))^(1/12)
    return f
end

friction_factor_churchill

## Definición de parámetros del gas y de la tubería

En esta sección definimos parámetros genéricos para el gas (por ejemplo, aire) y
la geometría de la tubería. Estos valores se reemplazarán según el ejercicio
que estemos resolviendo.

Para el aire tomaremos típicamente:

- $\gamma = 1.4$ (cociente de calores específicos),
- $R \approx 287\ \text{J/(kg⋅K)}$,
- $\mu$ (viscosidad dinámica) del orden de $1.8\times 10^{-5}\ \text{Pa.s}$ a temperatura ambiente.

In [7]:
# Parámetros del gas (ejemplo: aire)
γ   = 1.4          # adimensional
R   = 287.0        # J/(kg·K)
μ   = 1.8e-5       # Pa·s (aproximación a ~300 K)

# Geometría de la tubería (ejemplo genérico)
D   = 0.05         # m
L   = 20.0         # m
ε   = 4.5e-5       # m (rugosidad absoluta típica acero comercial)

A   = area_tubo(D) # m²

@show γ R μ D L ε A;

γ = 1.4
R = 287.0
μ = 1.8e-5
D = 0.05
L = 20.0
ε = 4.5e-5
A = 0.001963495408493621


## Modelo isotérmico en tubería

En el modelo isotérmico suponemos que la temperatura del gas se mantiene constante
a lo largo de la tubería:

$$ T_1 = T_2 = \cdots = T = \text{cte}. $$

Bajo este supuesto y usando las ecuaciones del formulario de flujo compresible,
las ecuaciones del problema pueden formularse en términos de variables
adimensionales (por ejemplo $X$, $F$, $z$) que dependen del caso:

- flujo en tramo de tubería entre dos presiones $P_1$ y $P_2$,
- entrada desde un reservorio a presión $P_0$,
- descarga hacia un reservorio, etc.

En este notebook dejaremos las rutinas de resolución con la estructura lista,
y completaremos las expresiones concretas (funciones residuo) a medida que
resolvamos ejercicios usando el formulario oficial.

In [ ]:
"""
    solve_isothermal_tube(P1, P2, T, D, L, ε; γ, R, μ, guess)

Resuelve el sistema de ecuaciones reducidas para flujo isotérmico
de un gas ideal en una tubería horizontal entre presiones P1 y P2.

La formulación concreta de las ecuaciones residuo se tomará del formulario
"Fórmulas de flujo compresible y Lapple" (versión del curso actual).

Por ahora, la función solo define la estructura y lanza un error
indicando que se deben completar las ecuaciones.
"""
function solve_isothermal_tube(P1, P2, T, D, L, ε;
                               γ = γ, R = R, μ = μ,
                               guess = [0.01, 1.1])

    F = P2 / P1            # cociente de presiones adimensional

    function residuals!(Fvars, res)
        # Ejemplo de variables adimensionales:
        # X, z = Fvars
        # Donde típicamente:
        #   X ~ (ṁ/A)^2 * (v1 / P1)   (según el formulario)
        #   z ~ v1 / v2
        #
        # Aquí deben implementarse las dos ecuaciones no lineales
        # (modelo isotérmico) en términos de X, z, F, f, L/D, etc.
        #
        # IMPORTANTE:
        #   1) Expresar f en función de Re (que depende de v).
        #   2) Cerrar el sistema usando la ecuación de estado y
        #      las definiciones adimensionales del formulario.
        #
        # Por ahora, dejamos las ecuaciones como TODO:
        res[1] = 0.0  # TODO: ecuación 1 isotérmica (adimensional)
        res[2] = 0.0  # TODO: ecuación 2 isotérmica (adimensional)
        error("TODO: completar las ecuaciones residuo para el modelo isotérmico.")
    end

    sol = nlsolve(residuals!, guess)
    X = sol.zero[1]
    z = sol.zero[2]

    # A partir de X y z se reconstruyen ṁ, v1, v2, ρ1, ρ2, etc.
    # Esto también se completará usando las definiciones del formulario.

    return (X = X, z = z, F = F)
end

## Modelo adiabático en tubería

En el modelo adiabático se supone:

- flujo estacionario de gas ideal,
- tubería horizontal,
- sin intercambio neto de calor con el entorno.

El balance de energía y el de cantidad de movimiento,
junto con la ecuación de estado, conducen —en el formulario del curso— a
sistemas de ecuaciones no lineales expresados en términos de variables
adimensionales como $X$, $F$, $z$, etc., que difieren según el caso:

- tramo de tubería entre $P_1$ y $P_2$,
- entrada desde un reservorio de presión $P_0$,
- posibles situaciones con flujo sónico, etc.

De nuevo, definimos primero la **estructura general** de la rutina en Julia
y completaremos las ecuaciones residuo caso a caso, según los ejercicios.

In [ ]:
"""
    solve_adiabatic_tube(P1, P2, T1, D, L, ε; γ, R, μ, guess)

Resuelve el sistema de ecuaciones reducidas para flujo adiabático
de un gas ideal en una tubería horizontal entre P1 y P2.

La formulación concreta de las ecuaciones residuo (en términos de X, z, F, etc.)
se tomará del formulario del curso. Por ahora, esta función es una plantilla.
"""
function solve_adiabatic_tube(P1, P2, T1, D, L, ε;
                              γ = γ, R = R, μ = μ,
                              guess = [0.01, 1.1])

    F = P2 / P1

    function residuals!(Fvars, res)
        # Variables adimensionales genéricas (a ajustar según el caso):
        # X, z = Fvars
        #
        # Aquí deben implementarse las ecuaciones adimensionales
        # del modelo adiabático, en la forma exacta del formulario.
        #
        # Incluye:
        #   - Ecuación de energía adiabática.
        #   - Ecuación de estado / relación geométrica / definición de X, z.
        #
        res[1] = 0.0  # TODO: ecuación 1 adiabática
        res[2] = 0.0  # TODO: ecuación 2 adiabática
        error("TODO: completar las ecuaciones residuo para el modelo adiabático.")
    end

    sol = nlsolve(residuals!, guess)
    X = sol.zero[1]
    z = sol.zero[2]

    # Reconstruir v1, v2, ṁ, ρ1, ρ2, etc. según definiciones del formulario.

    return (X = X, z = z, F = F)
end

## Caso especial: entrada desde un reservorio

Un caso típico en los ejercicios es el de un gas contenido en un **reservorio**
(lago de fluido) a presión $P_0$ y temperatura $T_0$, que descarga a una tubería:

- El gas en el reservorio se considera cuasi-estático y prácticamente en reposo.
- Se conoce $P_0$ (y habitualmente $T_0$).
- Aguas abajo se especifica:
  - ya sea una presión $P_3$,
  - o una longitud de tubería, o ambas,
  - y se determina el régimen de flujo (subsónico o crítico).

En el formulario, esto se traduce en otro sistema reducido con variables
adimensionales específicas (por ejemplo $X$, $F = P_3/P_0$, $z = v_0/v_3$, etc.).
Aquí dejaremos otra plantilla de función para este caso.

In [ ]:
"""
    solve_adiabatic_from_reservoir(P0, T0, P3, D, L, ε; γ, R, μ, guess)

Resuelve el problema de entrada desde un reservorio a una tubería,
bajo un modelo adiabático, según el sistema reducido del formulario
de flujo compresible. Por ahora, solo se define la estructura general.
"""
function solve_adiabatic_from_reservoir(P0, T0, P3, D, L, ε;
                                        γ = γ, R = R, μ = μ,
                                        guess = [0.01, 1.1])

    F = P3 / P0

    function residuals!(Fvars, res)
        # Variables adimensionales típicas (a ajustar):
        # X, z = Fvars
        #
        # Aquí irá el sistema adimensional propio del caso
        # "entrada desde reservorio" del formulario.
        #
        res[1] = 0.0  # TODO: ecuación 1 (adiabático desde reservorio)
        res[2] = 0.0  # TODO: ecuación 2
        error("TODO: completar las ecuaciones residuo para el caso de reservorio.")
    end

    sol = nlsolve(residuals!, guess)
    X = sol.zero[1]
    z = sol.zero[2]

    # Reconstruir ṁ, v en la entrada de la tubería, densidades, etc.

    return (X = X, z = z, F = F)
end

"""
    solve_adiabatic_from_reservoir(P0, T0, P3, D, L, ε; γ, R, μ, guess)

Resuelve el problema de entrada desde un reservorio a una tubería,
bajo un modelo adiabático, según el sistema reducido del formulario
de flujo compresible. Por ahora, solo se define la estructura general.
"""
function solve_adiabatic_from_reservoir(P0, T0, P3, D, L, ε;
                                        γ = γ, R = R, μ = μ,
                                        guess = [0.01, 1.1])

    F = P3 / P0

    function residuals!(Fvars, res)
        # Variables adimensionales típicas (a ajustar):
        # X, z = Fvars
        #
        # Aquí irá el sistema adimensional propio del caso
        # "entrada desde reservorio" del formulario.
        #
        res[1] = 0.0  # TODO: ecuación 1 (adiabático desde reservorio)
        res[2] = 0.0  # TODO: ecuación 2
        error("TODO: completar las ecuaciones residuo para el caso de reservorio.")
    end

    sol = nlsolve(residuals!, guess)
    X = sol.zero[1]
    z = sol.zero[2]

    # Reconstruir ṁ, v en la entrada de la tubería, densidades, etc.

    return (X = X, z = z, F = F)
end